# Global Macro Quant Research Notebook

**Date:** run date is printed automatically in Section 0
**Author:** Senior Quantitative Researcher

A four-layer quantitative signal stack applied to the most liquid global macro assets:
**SPY** (US equities), **BTC-USD** (crypto), **GC=F** (gold), **EURUSD=X** (FX).

| Layer | Model | Industry Standard |
|-------|-------|-------------------|
| 1 — Regime | Volatility Ratio + Candle Geometry | ✅ |
| 2 — Vol Forecast | HAR-OLS (Corsi 2009) | ✅ |
| 3 — Direction | XGBoost 3-class classifier | ✅ |
| 4 — Sizing | Half-Kelly (b=1 payoff) | ✅ |

---

## Section 0 — Config & Imports

In [1]:
# =============================================================
# SECTION 0 — Config & Imports
# =============================================================
import sys
import warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import httpx

# Add project root to path (for utils.py) and notebooks/ (for quant_helpers)
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(Path(".").resolve()))

from utils import load_config
import quant_helpers as qh

# Load config — uses load_config() which correctly handles allowed_chat_ids
# stored as a comma-separated string in config.toml
cfg = load_config(str(PROJECT_ROOT / "config.toml"))

TODAY = date.today().isoformat()
SEND_TO_TELEGRAM = True   # set False to preview only

warnings.filterwarnings("ignore")
print(f"Config loaded. Bot token: {'SET' if cfg.bot_token else 'MISSING'}")
print(f"Chat IDs: {cfg.allowed_chat_ids}")
print(f"Report date: {TODAY}")

01:16:47 | INFO    | alpha | Config loaded from D:\OMNP - Quant\Projetos\HedgePoly\prediction-market-analysis\config.toml


Config loaded. Bot token: SET
Chat IDs: [<REDACTED_CHAT>]
Report date: 2026-04-17


## Section 1 — Data Download & Quality Check

We use `yfinance` with `auto_adjust=True` to download 5 years of daily OHLCV bars.

| Ticker | Asset | Why |
|--------|-------|-----|
| SPY | US Equities ETF | Deepest global equity liquidity (~$500B AUM) |
| BTC-USD | Bitcoin | 24/7 global crypto market (~$1T market cap) |
| GC=F | Gold Futures | Universal safe-haven, ~$200B daily volume |
| EURUSD=X | EUR/USD FX | Most liquid FX pair (~$1.2T daily volume) |

> **Note (GC=F):** Gold futures have roll-date gaps. All return calculations use log-returns so any roll spike affects only a single observation.
> **Note (EURUSD=X):** Spot FX has no centralised volume — `volume = 0` is expected and ignored.

In [2]:
print("Downloading market data (5 years daily)...")
data = qh.load_all_assets(period="5y")

rows = []
for ticker, df in data.items():
    rows.append({
        "Ticker":     ticker,
        "Start":      str(df.index[0].date()),
        "End":        str(df.index[-1].date()),
        "Rows":       len(df),
        "NaN close":  int(df["close"].isna().sum()),
    })
pd.DataFrame(rows).set_index("Ticker")

,Start,End,Rows,NaN close
Ticker,,,,
SPY,2021-04-19,2026-04-16,1255,0
BTC-USD,2021-04-17,2026-04-17,1827,0
GC=F,2021-04-19,2026-04-17,1258,0
EURUSD=X,2021-04-19,2026-04-17,1300,0


## Section 2 — Feature Engineering

All features computed per asset independently.

| Feature | Formula | Literature |
|---------|---------|------------|
| VR (Volatility Ratio) | ATR(14) / close | Schwager, *Market Wizards* |
| VR_rank | Rolling 252-day percentile of VR | Ubiquitous regime filter |
| candle_det_norm | (open×close − high×low) / close² | Novel geometric feature ⚠️ |
| RV_1d | log_ret² | Andersen & Bollerslev (1998) |
| RV_5d / RV_22d | Rolling means of RV_1d | Corsi (2009) HAR-RV |
| mom_20 / mom_60 | log(close/close.shift(N)) | Jegadeesh & Titman (1993) |

> **Real-world adoption:** VR, HAR lags, and momentum are daily tools on systematic desks.
> Candle determinant (⚠️) is a novel/educational feature — interpret with caution.

In [3]:
print("Computing features...")
for ticker in data:
    data[ticker] = qh.add_features(data[ticker], ticker)

feature_cols = ["VR_rank", "candle_det_norm", "RV_1d", "RV_5d", "RV_22d", "mom_20", "mom_60"]
print("\nSPY features (last 5 rows):")
data["SPY"][feature_cols].tail()

Computing features...



SPY features (last 5 rows):


,VR_rank,candle_det_norm,RV_1d,RV_5d,RV_22d,mom_20,mom_60
Date,,,,,,,
2026-04-10,0.785714,0.000447,4.382435e-07,0.000138,0.000143,0.022646,-0.018115
2026-04-13,0.773810,0.000922,9.457478e-05,0.000152,0.000147,0.038047,-0.003463
2026-04-14,0.746032,-0.000128,1.466822e-04,0.000181,0.000143,0.040033,0.005929
2026-04-15,0.722222,0.001033,6.177997e-05,0.000067,0.000144,0.045266,0.014627
2026-04-16,0.710317,0.002017,6.023576e-06,0.000062,0.000140,0.061772,0.037649


## Section 3 — Layer 1: Regime Detection

> **Real-world adoption:** ✅ Volatility regime filters are standard on every systematic desk.

**Decision logic (VR_rank thresholds with candle determinant tiebreaker):**
- `VR_rank > 0.7` → **Trending** (high vol, directional)
- `VR_rank < 0.3` → **Choppy** (low vol, mean-reverting)
- Otherwise → **Ranging**
- Border bands (±0.05): candle_det_norm > 0 promotes to higher regime

**Reference:** Schwager, *Market Wizards* — volatility ratio as a regime filter.

In [4]:
print("Classifying regimes...")
for ticker in data:
    data[ticker]["regime"] = qh.classify_regimes_series(data[ticker])

print("\nCurrent regime per asset:")
for ticker in data:
    regime = data[ticker]["regime"].dropna().iloc[-1]
    print(f"  {ticker:<12} {regime}")

# 30-day regime history chart
fig = make_subplots(rows=2, cols=2, subplot_titles=list(data.keys()))
regime_colors = {"Trending": "green", "Ranging": "gold", "Choppy": "red"}
for i, (ticker, df) in enumerate(data.items(), 1):
    last30 = df["regime"].iloc[-30:]
    r, c = (i - 1) // 2 + 1, (i - 1) % 2 + 1
    for regime, color in regime_colors.items():
        mask = last30 == regime
        fig.add_trace(go.Bar(
            x=last30.index[mask], y=[1] * int(mask.sum()),
            name=regime, marker_color=color, showlegend=(i == 1),
        ), row=r, col=c)
fig.update_layout(title="30-Day Regime History", barmode="stack", height=500)
fig.show()

Classifying regimes...

Current regime per asset:
  SPY          Trending
  BTC-USD      Ranging
  GC=F         Trending
  EURUSD=X     Ranging


## Section 4 — Layer 2: Volatility Forecasting (HAR-OLS)

> **Real-world adoption:** ✅ HAR-RV (Corsi 2009, *Journal of Financial Econometrics*) is one of the most cited and widely used vol forecasting models in both academia and industry.

**Model:** OLS regression
`RV_next_day ~ β₀ + β₁·RV_1d + β₂·RV_5d + β₃·RV_22d`

**No lookahead:** Target is `RV_1d.shift(-1)`. Trained on first 80% of data only.
**vol_regime:** "Expanding" if forecast > 22-day rolling mean of RV; "Contracting" otherwise.

In [5]:
har_metrics = {}
print("Fitting HAR-OLS models (Corsi 2009)...")
for ticker in data:
    data[ticker], metrics = qh.fit_har_model(data[ticker], return_metrics=True)
    har_metrics[ticker] = metrics
    coef = metrics["coefficients"]
    print(f"\n{ticker}:")
    print(f"  R² = {metrics['r2']:.4f} | MAE = {metrics['mae']:.6f}")
    print(f"  β_1d={coef.get('RV_1d', coef.get('x1', 0)):.3f}  "
          f"β_5d={coef.get('RV_5d', coef.get('x2', 0)):.3f}  "
          f"β_22d={coef.get('RV_22d', coef.get('x3', 0)):.3f}")

print("\nCurrent vol regime per asset:")
for ticker in data:
    vr = data[ticker]["vol_regime"].dropna().iloc[-1]
    print(f"  {ticker:<12} {vr}")

Fitting HAR-OLS models (Corsi 2009)...



SPY:
  R² = 0.0870 | MAE = 0.000096
  β_1d=0.057  β_5d=0.306  β_22d=0.348

BTC-USD:
  R² = 0.0359 | MAE = 0.000746
  β_1d=0.053  β_5d=0.195  β_22d=0.229

GC=F:
  R² = 0.0268 | MAE = 0.000301
  β_1d=-0.071  β_5d=0.313  β_22d=0.251

EURUSD=X:
  R² = 0.0446 | MAE = 0.000022
  β_1d=0.020  β_5d=0.114  β_22d=0.517

Current vol regime per asset:
  SPY          Contracting
  BTC-USD      Expanding
  GC=F         Contracting
  EURUSD=X     Contracting


## Section 5 — Layer 3: Direction Signal (XGBoost)

> **Real-world adoption:** ✅ Gradient boosting is the dominant directional model on systematic equity and macro desks.

**Model:** `XGBClassifier` — 3 classes: Bearish (−1) / Neutral (0) / Bullish (+1)
**Target:** 20-day forward log-return terciles — boundaries from training set only (no lookahead)
**Features:** VR_rank, candle_det_norm, RV_1d, RV_5d, RV_22d, mom_20, mom_60
**Split:** 80/20 chronological, no shuffling. One model per asset.

In [6]:
# Snapshot regime and vol_regime BEFORE fit_xgb_signal truncates the DataFrame by 20 rows.
# fit_xgb_signal drops the last 20 rows (iloc[:-20]) to avoid boundary lookahead, so
# reading regime/vol_regime from data[ticker] after this call would return values ~20 days stale.
regime_snapshots     = {t: data[t]["regime"].dropna().iloc[-1] for t in data}
vol_regime_snapshots = {t: data[t]["vol_regime"].dropna().iloc[-1] for t in data}

xgb_meta = {}
print("Fitting XGBoost models (one per asset)...")
for ticker in data:
    data[ticker], meta = qh.fit_xgb_signal(data[ticker], return_meta=True)
    xgb_meta[ticker] = meta

print("\nCurrent direction signals:")
for ticker in data:
    row = data[ticker][["xgb_signal", "p_bearish", "p_neutral", "p_bullish"]].dropna().iloc[-1]
    print(f"  {ticker:<12} signal={int(row.xgb_signal):+d}  "
          f"P(Bear)={row.p_bearish:.2f}  P(Neut)={row.p_neutral:.2f}  P(Bull)={row.p_bullish:.2f}")

# Feature importance charts
fig = make_subplots(rows=2, cols=2, subplot_titles=list(data.keys()))
for i, (ticker, meta) in enumerate(xgb_meta.items(), 1):
    importances = meta["model"].feature_importances_
    r, c = (i - 1) // 2 + 1, (i - 1) % 2 + 1
    fig.add_trace(go.Bar(
        x=qh.FEATURE_COLS, y=importances,
        name=ticker, showlegend=False,
    ), row=r, col=c)
fig.update_layout(title="XGBoost Feature Importances by Asset", height=500)
fig.show()

Fitting XGBoost models (one per asset)...



Current direction signals:
  SPY          signal=-1  P(Bear)=0.80  P(Neut)=0.06  P(Bull)=0.14


  BTC-USD      signal=+1  P(Bear)=0.07  P(Neut)=0.11  P(Bull)=0.83
  GC=F         signal=+0  P(Bear)=0.40  P(Neut)=0.44  P(Bull)=0.17


  EURUSD=X     signal=+0  P(Bear)=0.25  P(Neut)=0.44  P(Bull)=0.32


## Section 6 — Layer 4: Position Sizing (Half-Kelly)

> **Real-world adoption:** ✅ Kelly Criterion (Kelly 1956) is used conceptually on every systematic desk. Half-Kelly is the standard practitioner variant.

**Formula (b=1 payoff — long/short pays ±1):**
```
f* = 2 × p_win − 1      (full Kelly)
kelly_f = f* × 0.5      (half-Kelly)
```
- **p_win** = max(P(Bullish), P(Bearish))
- **Floor:** 0% if p_win < 0.52
- **Cap:** 10% per asset
- p_win is always shown in the table even when position is FLAT

In [7]:
sizing = {}
print("Computing half-Kelly position sizes...")
for ticker in data:
    last = data[ticker][["p_bearish", "p_neutral", "p_bullish"]].dropna().iloc[-1]
    result = qh.compute_kelly(last.p_bullish, last.p_neutral, last.p_bearish)
    sizing[ticker] = result
    print(f"  {ticker:<12} side={result['side']:<6}  "
          f"P(Win)={result['p_win']*100:.0f}%  Kelly={result['kelly_pct']:.1f}%")

Computing half-Kelly position sizes...
  SPY          side=SHORT   P(Win)=80%  Kelly=10.0%


  BTC-USD      side=LONG    P(Win)=83%  Kelly=10.0%
  GC=F         side=FLAT    P(Win)=40%  Kelly=0.0%


  EURUSD=X     side=FLAT    P(Win)=32%  Kelly=0.0%


## Section 7 — Report

Aggregates all four layers into a final summary table, prints it in the notebook,
and sends to the Telegram group.

In [8]:
# Build signal rows — use snapshots for regime/vol_regime (taken before XGBoost truncated the df)
signals = []
for ticker in data:
    last_regime     = regime_snapshots[ticker]
    last_vol_regime = vol_regime_snapshots[ticker]
    sz = sizing[ticker]
    row = qh.build_summary_row(
        ticker     = ticker,
        regime     = last_regime,
        vol_regime = last_vol_regime,
        side       = sz["side"],
        p_win      = sz["p_win"],
        kelly_pct  = sz["kelly_pct"],
    )
    signals.append(row)

# Print summary table inside notebook
summary_text = qh.build_summary_table_text(signals, date_str=TODAY)
print(summary_text)

# Build Telegram HTML
html_report = qh.build_telegram_html(signals, date_str=TODAY)

# Send to Telegram
if SEND_TO_TELEGRAM and cfg.bot_token and cfg.allowed_chat_ids:
    url = f"https://api.telegram.org/bot{cfg.bot_token}/sendMessage"
    chunks = []
    current = ""
    for line in html_report.split("\n"):
        if len(current) + len(line) + 1 > 4000:
            if current:
                chunks.append(current)
            current = line
        else:
            current += ("\n" if current else "") + line
    if current:
        chunks.append(current)

    with httpx.Client(timeout=15.0) as client:
        for chat_id in cfg.allowed_chat_ids:
            for i, chunk in enumerate(chunks, 1):
                resp = client.post(url, json={
                    "chat_id":                  chat_id,
                    "text":                     chunk,
                    "parse_mode":               "HTML",
                    "disable_web_page_preview": True,
                })
                status = "OK" if resp.status_code == 200 else f"ERROR {resp.status_code}: {resp.text[:100]}"
                print(f"  chat_id={chat_id}  chunk={i}/{len(chunks)}  -> {status}")
else:
    print("\n[PREVIEW MODE — Telegram not configured or SEND_TO_TELEGRAM=False]")
    print(html_report)


 GLOBAL MACRO QUANT REPORT — 2026-04-17

Asset      Regime     Vol Regime    Signal   P(Win)  Kelly%
-------------------------------------------------------
SPY        Trending   Contracting   BEARISH     80%   10.0%
BTC-USD    Ranging    Expanding     BULLISH     83%   10.0%
GC=F       Trending   Contracting   NEUTRAL     40%    0.0%
EURUSD=X   Ranging    Contracting   NEUTRAL     32%    0.0%
-------------------------------------------------------


01:17:00 | INFO    | httpx | HTTP Request: POST https://api.telegram.org/bot<REDACTED_TOKEN>/sendMessage "HTTP/1.1 200 OK"


  chat_id=<REDACTED_CHAT>  chunk=1/1  -> OK
